# nseos - Colab driver (v2)

Thin front-end: every heavy step is a cached stage of the `nseos` package.
Re-running a cell reuses finished outputs on Drive; add `--force` to recompute one stage.

**Private repo:** add a GitHub token with read access to this repo as a Colab secret named `GH_TOKEN` (key icon in the left bar).

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/ns_eos_project'   # artefacts persist here
os.environ['NSEOS_ROOT'] = ROOT
token = userdata.get('GH_TOKEN')
if not os.path.exists('/content/ns-eos-amortized'):
    !git clone -q https://{token}@github.com/sohl-ar/ns-eos-amortized.git /content/ns-eos-amortized
else:
    !git -C /content/ns-eos-amortized pull -q
%cd /content/ns-eos-amortized
!pip install -q -e .

In [ ]:
!python -m pytest tests -q   # likelihood == simulator marginal; simulator properties

## Core pipeline

In [ ]:
!python -m nseos.cli simulate

In [ ]:
!python -m nseos.cli train

In [ ]:
!python -m nseos.cli evaluate

## Real data and nested-sampling reference

In [ ]:
!python -m nseos.cli grid realdata reference-real compare-real

## Reviewer analyses (long; each is resumable)

In [ ]:
!python -m nseos.cli identifiability zero-noise

In [ ]:
!python -m nseos.cli ablate-arch   # 3 aggregators x 3 seeds; reduce with --set ablation.seeds=[0]

In [ ]:
!python -m nseos.cli ablate-messenger

In [ ]:
!python -m nseos.cli reference-synth   # CPU-heavy: dynesty on 24 synthetic catalogs

In [ ]:
!python -m nseos.cli importance convergence   # exact IS correction; TOV convergence

In [ ]:
!python -m nseos.cli economics systematics summary

## Results

In [ ]:
from IPython.display import Image, Markdown, display
from nseos.config import Paths, load_config
from nseos.provenance import FIGURE_STAGE
from pathlib import Path
paths = Paths(load_config())
display(Markdown(Path(paths.result('paper_numbers', 'md')).read_text()))
for name in FIGURE_STAGE:
    f = Path(paths.figure(name) + '.png')
    if f.exists(): display(Image(str(f), width=900))

In [ ]:
# Optional: release the GPU when finished
# from google.colab import runtime; runtime.unassign()